# Evaluation & Visualization
This notebook provides utility functions for testing and comparing Steiner tree algorithms:
1. **Single algorithm testing** — run one algorithm on a set of instances and print detailed results
2. **Comparison table** — run all algorithms side-by-side, show weights and deviation from optimum
3. **Parallel comparison** — same as above but using multi-threading/processing
4. **Visualization** — bar charts for weights, times, errors, and a speed-vs-quality scatter plot

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import time
from itertools import combinations
from pathlib import Path

# Load algorithms and STP parser from the algorithms notebook
%run 01_algorithms.ipynb

## Single Algorithm Testing & Sequential Comparison

In [ ]:
def test_algorithm(algorithm, files, is_exact=False, max_nodes=30):
    """Run a single algorithm on each .stp file and print detailed results."""
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        print(f'--- {name} ---')
        print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Terminals: {len(terminals)}')

        # Skip large instances for exact (brute-force) algorithms
        if is_exact and G.number_of_nodes() > max_nodes:
            print('Skipped (too many nodes for exact algorithm)')
            print()
            continue

        start = time.time()
        if is_exact:
            tree, weight, subsets = algorithm(G, terminals)
            print(f'Subsets checked: {subsets}')
        else:
            tree, weight = algorithm(G, terminals)
        elapsed = time.time() - start

        steiner_points = [v for v in tree.nodes() if v not in terminals]
        print(f'Weight: {weight}')
        print(f'Steiner points used: {steiner_points}')
        print(f'Edges: {list(tree.edges(data=True))}')
        print(f'Time: {elapsed:.4f}s')
        print()


def compare_algorithms(algorithms, files, bf_max_nodes=25):
    """Run all algorithms on each instance and print a comparison table.

    The first algorithm in the list is treated as the baseline (typically brute force).
    Other algorithms show '*' if they match the optimum, or the deviation in %.
    """
    results = []
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        row = {'instance': name, 'nodes': G.number_of_nodes(), 'terminals': len(terminals)}

        for alg_name, alg_func, is_exact in algorithms:
            # Skip exact algorithms on large instances
            if is_exact and G.number_of_nodes() > bf_max_nodes:
                row[alg_name] = None
                row[alg_name + '_time'] = None
                continue

            start = time.time()
            if is_exact:
                tree, weight, _ = alg_func(G, terminals)
            else:
                tree, weight = alg_func(G, terminals)
            elapsed = time.time() - start
            row[alg_name] = weight
            row[alg_name + '_time'] = elapsed

        results.append(row)

    # --- Print formatted table ---
    alg_names = [name for name, _, _ in algorithms]
    header = f"{'Instance':<10} {'N':>4} {'T':>4} | " + "  ".join(f"{name[:6]:>6}" for name in alg_names)
    print(header)
    print('-' * len(header))

    bf_name = alg_names[0]  # baseline algorithm (first in list)
    for r in results:
        bf = r[bf_name]
        if bf is None:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} |      -", end='')
        else:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} | {bf:>6.0f}", end='')

        for alg_name in alg_names[1:]:
            w = r[alg_name]
            if bf is not None and w == bf:
                print(f"  {'*':>6}", end='')       # matches optimum
            elif bf is not None:
                diff = (w - bf) / bf * 100          # % deviation from optimum
                print(f"  {w:>3.0f}+{diff:.0f}%", end='')
            else:
                print(f"  {w:>6.0f}", end='')
        print()

    print()
    print("* = optimalno rešenje, - = preskočeno (preveliko za brute force)")
    return results


def plot_times(results, algorithms):
    """Bar chart: execution time per algorithm per instance (log scale)."""
    alg_names = [name for name, _, _ in algorithms]
    instances = [r['instance'] for r in results]
    x = range(len(instances))
    width = 0.15
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, alg_name in enumerate(alg_names):
        times = []
        for r in results:
            t = r.get(alg_name + '_time')
            times.append(t if t is not None else 0)
        offset = (i - len(alg_names) / 2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], times, width, label=alg_name, color=colors[i % len(colors)])

    ax.set_xlabel('Instanca')
    ax.set_ylabel('Vreme (s)')
    ax.set_title('Vreme izvršavanja po algoritmu')
    ax.set_xticks(list(x))
    ax.set_xticklabels(instances, rotation=45, ha='right')
    ax.legend()
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

## Parallel Comparison
Runs algorithm-instance pairs concurrently.  
- **Linux/macOS** — uses `ProcessPoolExecutor` with fork context  
- **Windows** — falls back to `ThreadPoolExecutor` (notebook functions can't be pickled for spawn)

In [ ]:
import time
import os
import sys
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

# Global registry so worker processes/threads can look up algorithm functions by name
_PAR_ALG_FUNCS = {}


def _par_worker(task):
    """Worker function executed in parallel for each (instance, algorithm) pair."""
    idx, filepath, alg_name, is_exact, bf_max_nodes = task
    G, terminals, name = parse_stp(filepath)
    n = G.number_of_nodes()
    t = len(terminals)

    if is_exact and n > bf_max_nodes:
        return (idx, name, n, t, alg_name, None, None)

    func = _PAR_ALG_FUNCS[alg_name]
    start = time.time()
    if is_exact:
        tree, weight, _ = func(G, terminals)
    else:
        tree, weight = func(G, terminals)
    elapsed = time.time() - start
    return (idx, name, n, t, alg_name, weight, elapsed)


def _print_compare_table(results, algorithms):
    """Print formatted comparison table (shared by sequential and parallel versions)."""
    alg_names = [name for name, _, _ in algorithms]
    header = f"{'Instance':<10} {'N':>4} {'T':>4} | " + "  ".join(f"{name[:6]:>6}" for name in alg_names)
    print(header)
    print('-' * len(header))

    bf_name = alg_names[0]  # baseline
    for r in results:
        bf = r.get(bf_name)
        if bf is None:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} |      -", end='')
        else:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} | {bf:>6.0f}", end='')
        for alg_name in alg_names[1:]:
            w = r.get(alg_name)
            if bf is not None and w == bf:
                print(f"  {'*':>6}", end='')
            elif bf is not None:
                diff = (w - bf) / bf * 100
                print(f"  {w:>3.0f}+{diff:.0f}%", end='')
            else:
                print(f"  {w:>6.0f}", end='')
        print()
    print()
    print("* = optimalno resenje, - = preskoceno (preveliko za brute force)")


def compare_algorithms_parallel(algorithms, files, bf_max_nodes=25,
                                n_workers=None, print_table=True):
    """Parallel version of compare_algorithms.

    Distributes (instance x algorithm) pairs across workers and collects results.
    """
    global _PAR_ALG_FUNCS
    _PAR_ALG_FUNCS = {name: func for name, func, _ in algorithms}

    files = list(files)

    # Build task list: one task per (instance, algorithm) pair
    tasks = []
    for idx, filepath in enumerate(files):
        for alg_name, alg_func, is_exact in algorithms:
            tasks.append((idx, str(filepath), alg_name, is_exact, bf_max_nodes))

    rows = [dict() for _ in files]

    # Choose executor based on platform
    if sys.platform == 'win32':
        Executor = ThreadPoolExecutor
        ctx_kwargs = {}
    else:
        Executor = ProcessPoolExecutor
        ctx_kwargs = {'mp_context': mp.get_context('fork')}

    # Execute all tasks and collect results with progress indicator
    with Executor(max_workers=n_workers, **ctx_kwargs) as ex:
        done = 0
        total = len(tasks)
        for idx, name, n, t, alg_name, weight, elapsed in ex.map(_par_worker, tasks):
            r = rows[idx]
            r['instance'] = name
            r['nodes'] = n
            r['terminals'] = t
            r[alg_name] = weight
            r[alg_name + '_time'] = elapsed
            done += 1
            print(f'\r[{done}/{total}] {name} :: {alg_name}            ', end='', flush=True)
    print()

    if print_table:
        _print_compare_table(rows, algorithms)
    return rows

## Visualization: Weights, Errors & Execution Times

In [ ]:
def plot_weights(results, algorithms):
    """Bar chart: Steiner tree weight per algorithm per instance."""
    alg_names = [name for name, _, _ in algorithms]
    instances = [r['instance'] for r in results]
    x = range(len(instances))
    width = 0.15
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, alg_name in enumerate(alg_names):
        weights = []
        for r in results:
            w = r.get(alg_name)
            weights.append(w if w is not None else 0)
        offset = (i - len(alg_names) / 2 + 0.5) * width
        ax.bar([xi + offset for xi in x], weights, width, label=alg_name, color=colors[i % len(colors)])

    ax.set_xlabel('Instanca')
    ax.set_ylabel('Težina Steinerovog stabla')
    ax.set_title('Poređenje težina po algoritmu')
    ax.set_xticks(list(x))
    ax.set_xticklabels(instances, rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_errors(results, algorithms):
    """Bar chart: % deviation from brute-force optimum, per instance.

    Only instances where brute-force results exist are shown.
    """
    alg_names = [name for name, _, _ in algorithms]
    bf_name = alg_names[0]

    # Filter to instances where we have the baseline (brute-force) result
    filtered = [r for r in results if r.get(bf_name) is not None]
    if not filtered:
        print("Nema instanci sa brute force za poređenje grešaka.")
        return

    instances = [r['instance'] for r in filtered]
    x = range(len(instances))
    width = 0.18
    colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, alg_name in enumerate(alg_names[1:]):  # skip brute-force itself
        errors = []
        for r in filtered:
            bf = r[bf_name]
            w = r[alg_name]
            errors.append((w - bf) / bf * 100)
        offset = (i - (len(alg_names) - 1) / 2 + 0.5) * width
        ax.bar([xi + offset for xi in x], errors, width, label=alg_name, color=colors[i % len(colors)])

    ax.set_xlabel('Instanca')
    ax.set_ylabel('Greška (%)')
    ax.set_title('Odstupanje od optimuma (Brute Force)')
    ax.set_xticks(list(x))
    ax.set_xticklabels(instances, rotation=45, ha='right')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()


def print_times(results, algorithms):
    """Print a table of execution times per algorithm per instance."""
    alg_names = [name for name, _, _ in algorithms]
    header = f"{'Instance':<10} {'N':>4} {'T':>4} | " + "  ".join(f"{name[:8]:>10}" for name in alg_names)
    print(header)
    print('-' * len(header))
    for r in results:
        print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} |", end='')
        for alg_name in alg_names:
            t = r.get(alg_name + '_time')
            if t is None:
                print(f"  {'-':>10}", end='')
            else:
                print(f"  {t:>9.4f}s", end='')
        print()

## Summary Charts: Average Error & Speed vs. Quality

In [ ]:
def plot_avg_errors(results, algorithms):
    """Bar chart: average % error across all instances (vs brute-force optimum)."""
    alg_names = [name for name, _, _ in algorithms]
    bf_name = alg_names[0]

    filtered = [r for r in results if r.get(bf_name) is not None]
    if not filtered:
        print("Nema instanci sa brute force za poređenje grešaka.")
        return

    colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    fig, ax = plt.subplots(figsize=(8, 5))

    for i, alg_name in enumerate(alg_names[1:]):
        errors = [(r[alg_name] - r[bf_name]) / r[bf_name] * 100 for r in filtered]
        avg_err = sum(errors) / len(errors)
        ax.bar(alg_name, avg_err, color=colors[i % len(colors)])
        ax.text(i, avg_err + 0.3, f'{avg_err:.1f}%', ha='center', fontweight='bold')

    ax.set_ylabel('Prosečna greška (%)')
    ax.set_title('Prosečna greška u odnosu na optimum')
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()


def plot_speed_vs_quality(results, algorithms):
    """Scatter plot: average time (x, log scale) vs average error (y) per algorithm.

    Each dot represents one algorithm, summarized across all instances
    where brute-force results are available.
    """
    alg_names = [name for name, _, _ in algorithms]
    bf_name = alg_names[0]

    filtered = [r for r in results if r.get(bf_name) is not None]
    if not filtered:
        print("Nema instanci sa brute force za poređenje.")
        return

    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, alg_name in enumerate(alg_names):
        # Gather times and errors only for instances where this algorithm ran
        times = [r[alg_name + '_time'] for r in filtered if r.get(alg_name + '_time') is not None]
        errors = []
        for r in filtered:
            bf = r[bf_name]
            w = r.get(alg_name)
            if w is not None and r.get(alg_name + '_time') is not None:
                errors.append((w - bf) / bf * 100)

        if times and errors:
            avg_time = sum(times) / len(times)
            avg_err = sum(errors) / len(errors)
            ax.scatter(avg_time, avg_err, s=150, color=colors[i % len(colors)], zorder=5)
            ax.annotate(alg_name, (avg_time, avg_err), textcoords="offset points",
                       xytext=(10, 5), fontsize=10)

    ax.set_xlabel('Prosečno vreme (s)')
    ax.set_ylabel('Prosečna greška (%)')
    ax.set_title('Brzina vs. kvalitet')
    ax.set_xscale('log')
    ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def print_weight_ratios(results, algorithms):
    """Print weight ratio (algorithm / optimum) per instance, plus average ratio.

    Ratio of 1.0 = optimal, >1.0 = suboptimal.
    """
    alg_names = [name for name, _, _ in algorithms]
    bf_name = alg_names[0]

    filtered = [r for r in results if r.get(bf_name) is not None]
    if not filtered:
        print("Nema instanci sa brute force za poređenje.")
        return

    header = f"{'Instance':<10} {'OPT':>5} | " + "  ".join(f"{name[:8]:>10}" for name in alg_names[1:])
    print(header)
    print('-' * len(header))
    for r in filtered:
        bf = r[bf_name]
        print(f"{r['instance']:<10} {bf:>5.0f} |", end='')
        for alg_name in alg_names[1:]:
            w = r[alg_name]
            ratio = w / bf
            print(f"  {ratio:>10.4f}", end='')
        print()

    # Print average approximation ratios
    print()
    print("Prosečni odnosi:")
    for alg_name in alg_names[1:]:
        ratios = [r[alg_name] / r[bf_name] for r in filtered]
        avg = sum(ratios) / len(ratios)
        print(f"  {alg_name}: {avg:.4f}")